# 윈도우 라벨 매핑

CloudTrail 이벤트를 **5개씩 윈도우로 분할**할 때, 각 윈도우에 전술(tactic)·기법(technique) 라벨을 어떻게 붙일지 정리한다.

- 이벤트 단위 라벨: 각 이벤트는 `benign`이거나 하나의 (전술, 기법) 라벨을 가진다.
- 윈도우 단위 라벨: 윈도우 안의 5개 이벤트 라벨을 모아 윈도우 라벨을 만든다.

## 방식 1. 공격 이벤트가 하나라도 있으면 공격 라벨

윈도우 안에 **공격 이벤트가 하나라도 있으면** 그 윈도우 전체를 공격으로 라벨링한다.

- 공격 이벤트가 없으면 → `benign`
- 공격 이벤트가 1개 이상이면 → 그 이벤트의 전술·기법 라벨을 윈도우 라벨로 사용
- 서로 다른 기법이 여러 개 섞여 있으면 → 등장한 기법(과 전술)을 모두 포함하는 **멀티라벨**

예시 (윈도우 크기 5):

| 윈도우 | 이벤트 라벨 | 윈도우 라벨 |
|---|---|---|
| w0 | benign ×5 | benign |
| w1 | benign ×4, T1078 | TA0001 / T1078 |
| w2 | T1078, benign ×3, T1530 | {TA0001, TA0009} / {T1078, T1530} |

**장점**
- 공격을 놓치지 않는다 (recall 관점에서 유리).
- 규칙이 단순해서 구현과 설명이 쉽다.

**단점**
- 공격 이벤트가 1개뿐이고 나머지 4개가 정상이어도 공격으로 라벨링되어, 윈도우 텍스트 대부분이 정상 행위인 노이즈 라벨이 생긴다.
- 슬라이딩 윈도우(stride < 5)를 쓰면 같은 공격 이벤트가 여러 윈도우에 공격 라벨을 퍼뜨려 공격 윈도우 수가 부풀려진다.

## 예시 코드

In [1]:
import pandas as pd

# 이벤트 단위 라벨 예시 (benign이면 tactic/technique/attack_id = None)
# attack_id: 한 번의 기법 실행을 이루는 이벤트 묶음 ID (방식 2에서 사용)
events = pd.DataFrame({
    "eventName": [
        "DescribeInstances", "ListBuckets", "GetCallerIdentity", "DescribeRegions", "ListUsers",
        "DescribeInstances", "ListBuckets", "GetCallerIdentity", "DescribeRegions", "ConsoleLogin",
        "ConsoleLogin", "ListUsers", "DescribeRegions", "ListBuckets", "GetObject",
    ],
    "tactic":    [None]*9 + ["TA0001", "TA0001", None, None, None, "TA0009"],
    "technique": [None]*9 + ["T1078",  "T1078",  None, None, None, "T1530"],
    "attack_id": [None]*9 + ["a1",     "a1",     None, None, None, "a2"],
})
events

,eventName,tactic,technique,attack_id
0,DescribeInstances,NaN,NaN,NaN
1,ListBuckets,NaN,NaN,NaN
2,GetCallerIdentity,NaN,NaN,NaN
3,DescribeRegions,NaN,NaN,NaN
4,ListUsers,NaN,NaN,NaN
5,DescribeInstances,NaN,NaN,NaN
6,ListBuckets,NaN,NaN,NaN
7,GetCallerIdentity,NaN,NaN,NaN
8,DescribeRegions,NaN,NaN,NaN
9,ConsoleLogin,TA0001,T1078,a1


In [2]:
def make_windows(df, size=5, stride=5):
    """이벤트를 size개씩 묶는다. stride < size이면 슬라이딩 윈도우."""
    return [df.iloc[i:i + size] for i in range(0, len(df) - size + 1, stride)]


def label_any_attack(window):
    """방식 1: 공격 이벤트가 하나라도 있으면 등장한 전술·기법을 모두 윈도우 라벨로."""
    attack = window.dropna(subset=["technique"])
    if attack.empty:
        return {"is_attack": False, "tactics": [], "techniques": []}
    return {
        "is_attack": True,
        "tactics": sorted(attack["tactic"].unique()),
        "techniques": sorted(attack["technique"].unique()),
    }

In [3]:
rows = []
for i, w in enumerate(make_windows(events, size=5, stride=5)):
    rows.append({"window": f"w{i}", "n_attack_events": w["technique"].notna().sum(), **label_any_attack(w)})
pd.DataFrame(rows)

,window,n_attack_events,is_attack,tactics,techniques
0,w0,0,False,[],[]
1,w1,1,True,[TA0001],[T1078]
2,w2,2,True,"[TA0001, TA0009]","[T1078, T1530]"


슬라이딩 윈도우(stride=1)에서는 같은 공격 이벤트가 여러 윈도우에 걸쳐 공격 라벨을 만든다.

In [4]:
rows = []
for i, w in enumerate(make_windows(events, size=5, stride=1)):
    rows.append({"window": f"w{i}", "n_attack_events": w["technique"].notna().sum(), **label_any_attack(w)})
sliding = pd.DataFrame(rows)
print("공격 윈도우 비율:", sliding["is_attack"].mean().round(2))
sliding

공격 윈도우 비율: 0.55


,window,n_attack_events,is_attack,tactics,techniques
0,w0,0,False,[],[]
1,w1,0,False,[],[]
2,w2,0,False,[],[]
3,w3,0,False,[],[]
4,w4,0,False,[],[]
5,w5,1,True,[TA0001],[T1078]
6,w6,2,True,[TA0001],[T1078]
7,w7,2,True,[TA0001],[T1078]
8,w8,2,True,[TA0001],[T1078]
9,w9,2,True,[TA0001],[T1078]


## 방식 2. 기법 이벤트 전체가 윈도우 안에 있어야 공격 라벨

하나의 기법 실행(`attack_id`)을 이루는 공격 이벤트가 **잘리지 않고 모두 윈도우 안에 들어 있을 때만** 그 전술·기법을 윈도우 라벨로 붙인다.

- 윈도우 안에 완전히 포함된 기법이 **하나라도 있으면** → 공격 (포함된 기법들의 전술·기법이 라벨, 여러 개면 멀티라벨)
- 완전히 포함된 기법이 **하나도 없으면** → `benign`
  - 공격 이벤트가 윈도우 경계에 걸쳐 일부만 들어 있는 경우도 `benign`으로 본다.

예시 (윈도우 크기 5, 기법 a1 = T1078 이벤트 2개가 w1·w2 경계에 걸침, a2 = T1530 이벤트 1개):

| 윈도우 | 이벤트 라벨 | 방식 1 | 방식 2 |
|---|---|---|---|
| w0 | benign ×5 | benign | benign |
| w1 | benign ×4, a1 일부 | TA0001 / T1078 | benign (a1이 잘림) |
| w2 | a1 일부, benign ×3, a2 | {TA0001, TA0009} / {T1078, T1530} | TA0009 / T1530 |

**장점**
- 라벨이 붙은 윈도우에는 그 기법의 행위가 온전히 들어 있어, 윈도우 텍스트와 라벨이 잘 맞는다 (라벨 정밀도 높음).
- 슬라이딩 윈도우에서도 기법 하나가 공격 라벨을 퍼뜨리는 범위가 줄어든다.

**단점**
- 이벤트 수가 5개보다 많은 기법은 어떤 윈도우에도 완전히 들어갈 수 없어 **영원히 공격으로 라벨링되지 않는다.** 윈도우 크기를 정하기 전에 기법별 이벤트 수 분포를 확인해야 한다.
- 경계에 걸친 공격 이벤트가 있는 윈도우가 `benign`으로 들어가, 정상 데이터에 공격 행위가 섞인다.
- 기법 실행 단위(`attack_id`)를 이벤트에 미리 붙여 두어야 한다.

In [5]:
def label_fully_contained(window, all_events):
    """방식 2: 이벤트 전체가 윈도우 안에 들어온 기법만 윈도우 라벨로."""
    total = all_events["attack_id"].value_counts()   # 기법 실행별 전체 이벤트 수
    in_win = window["attack_id"].value_counts()      # 윈도우 안에 들어온 이벤트 수
    full_ids = [a for a, n in in_win.items() if n == total[a]]
    attack = window[window["attack_id"].isin(full_ids)]
    if attack.empty:
        return {"is_attack": False, "tactics": [], "techniques": []}
    return {
        "is_attack": True,
        "tactics": sorted(attack["tactic"].unique()),
        "techniques": sorted(attack["technique"].unique()),
    }

방식 1과 방식 2를 같은 윈도우에서 비교한다.

In [6]:
def compare(df, size=5, stride=5):
    rows = []
    for i, w in enumerate(make_windows(df, size=size, stride=stride)):
        m1, m2 = label_any_attack(w), label_fully_contained(w, df)
        rows.append({
            "window": f"w{i}",
            "m1_techniques": m1["techniques"] or "benign",
            "m2_techniques": m2["techniques"] or "benign",
        })
    return pd.DataFrame(rows)

compare(events, stride=5)

,window,m1_techniques,m2_techniques
0,w0,benign,benign
1,w1,[T1078],benign
2,w2,"[T1078, T1530]",[T1530]


In [7]:
# 슬라이딩 윈도우(stride=1)
cmp = compare(events, stride=1)
print("공격 윈도우 수 — 방식 1:", (cmp["m1_techniques"] != "benign").sum(),
      "/ 방식 2:", (cmp["m2_techniques"] != "benign").sum())
cmp

공격 윈도우 수 — 방식 1: 6 / 방식 2: 5


,window,m1_techniques,m2_techniques
0,w0,benign,benign
1,w1,benign,benign
2,w2,benign,benign
3,w3,benign,benign
4,w4,benign,benign
5,w5,[T1078],benign
6,w6,[T1078],[T1078]
7,w7,[T1078],[T1078]
8,w8,[T1078],[T1078]
9,w9,[T1078],[T1078]


In [8]:
# 기법 실행별 이벤트 수: 윈도우 크기(5)보다 큰 기법은 방식 2에서 라벨이 붙지 않는다
sizes = events.dropna(subset=["attack_id"]).groupby(["attack_id", "technique"]).size().rename("n_events")
sizes.to_frame().assign(fits_in_window=sizes <= 5)

,,n_events,fits_in_window
attack_id,technique,,
a1,T1078,2,True
a2,T1530,1,True


## 실제 데이터에 적용 (방식 1)

`src/windows.py`와 같은 코드를 **방식 1**로만 실제 데이터에 적용한다.

- 정책: 크기 5, stride 1, 방식 1 (공격 이벤트가 하나라도 있으면 공격)
- 입력: `notebooks/preprocess.ipynb`가 만든 두 버전
  - **A**: 압축 안 한 이벤트(`events_A.csv`)를 분할한 뒤, 내용이 같은 윈도우를 합친다
  - **B**: 반복 압축한 이벤트(`events_B.csv`)를 분할한다
- 세션(원본 파일)마다 시간순으로 정렬해서 따로 분할한다. 세션끼리 이어 붙이지 않는다.
- `session`, 시간, `ip`, `ip_kind`, 라벨은 **정렬·분할까지만** 쓰고 윈도우 분할 후에는 버린다 (`src/windows.py`의 `model_input`). 모델 입력에는 들어가지 않는다.
- 위 예시의 `make_windows`와 달리, 끝에 남는 5개 미만 이벤트도 윈도우 하나로 남긴다 (이벤트가 1~4개뿐인 Stratus 실행이 사라지지 않게).

In [9]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
INTERIM = ROOT / "data" / "interim"
INPUTS = {"A": INTERIM / "events_A.csv", "B": INTERIM / "events_B.csv"}
SIZE, STRIDE = 5, 1

# 윈도우 내용 비교에 쓰는 모델 입력 필드 (IP, 세션, 시간, 라벨은 제외)
SIGNATURE_COLS = ["eventSource", "eventName", "errorCode", "readOnly", "userType", "region_changed", "repeat_count"]


def load_events(path):
    events = pd.read_csv(path, keep_default_na=False, na_values={"attack_id": [""], "technique": [""]})
    events["is_attack"] = events["is_attack"].astype(str) == "True"
    return events.sort_values(["session", "eventTime", "seq"], kind="stable")


def make_windows_keep_tail(df, size=5, stride=1):
    """이벤트를 size개씩 묶고, 끝에 남는 size개 미만 이벤트도 윈도우로 남긴다."""
    starts = list(range(0, max(len(df) - size, 0) + 1, stride))
    if starts[-1] + size < len(df):
        starts.append(starts[-1] + stride)
    return [df.iloc[i:i + size] for i in starts]


def label_window(window):
    """방식 1: 공격 이벤트가 하나라도 있으면 등장한 전술·기법을 모두 윈도우 라벨로."""
    attack = window[window["is_attack"]]
    if attack.empty:
        return {"is_attack": False, "tactics": "", "techniques": ""}
    tactics = sorted({t for ts in attack["tactics"] for t in ts.split("|") if t})
    return {"is_attack": True, "tactics": "|".join(tactics),
            "techniques": "|".join(sorted(attack["technique"].unique()))}


def window_signature(window):
    """모델 입력 필드만으로 윈도우 내용을 문자열로 만든다.
    actor는 세션마다 번호가 다르므로 윈도우 안에서 처음 나온 순서로 다시 번호를 매긴다."""
    actor = pd.factorize(window["actor"])[0]
    ev = window[SIGNATURE_COLS].astype(str).agg("|".join, axis=1)
    return " > ".join(f"{e}|actor{a}" for e, a in zip(ev, actor))

In [10]:
def build_windows(events, size=5, stride=1):
    rows = []
    for session, g in events.groupby("session", sort=False):
        for i, w in enumerate(make_windows_keep_tail(g, size, stride)):
            rows.append({
                "session": session, "source": w["source"].iat[0], "window": i,
                "n_events": len(w), "n_attack_events": int(w["is_attack"].sum()),
                "event_uids": "|".join(w["event_uid"]),
                "eventNames": " > ".join(w["eventName"]),
                "signature": window_signature(w),
                **label_window(w),
            })
    return pd.DataFrame(rows)


def mark_duplicates(win):
    """내용(signature)과 라벨이 같은 윈도우를 한 묶음으로 표시한다. 윈도우는 지우지 않는다."""
    win = win.copy()
    label = win["techniques"].where(win["is_attack"], "BENIGN")
    key = win["signature"] + "#" + label
    win["dup_group"] = pd.factorize(key)[0]
    win["dup_count"] = win.groupby("dup_group")["dup_group"].transform("size")
    win["is_first"] = ~key.duplicated()                                   # A의 학습 데이터는 이것만 쓴다
    win["label_conflict"] = label.groupby(win["signature"]).transform("nunique") > 1
    return win


windows = {v: mark_duplicates(build_windows(load_events(path), SIZE, STRIDE)) for v, path in INPUTS.items()}

### A와 B 비교

- A의 학습 데이터는 같은 윈도우를 합친 것(`is_first`), B는 전체 윈도우를 쓴다.
- 평가 데이터는 두 버전 모두 실제 분포를 보도록 합치지 않고 전부 쓴다.
- `label_conflict`: 내용은 같은데 라벨이 다른 윈도우. 공격자와 정상 사용자가 같은 조회를 한 경우가 대부분이다.

In [11]:
def counts(w):
    n_att = int(w["is_attack"].sum())
    return {"윈도우": len(w), "정상": len(w) - n_att, "공격": n_att}


summary = pd.DataFrame({
    "A 전체": counts(windows["A"]),
    "A 합친 뒤 (학습)": counts(windows["A"][windows["A"]["is_first"]]),
    "B 전체 (학습)": counts(windows["B"]),
    "B 합친 뒤": counts(windows["B"][windows["B"]["is_first"]]),
}).T
summary["라벨 충돌"] = [int(windows[v[0]].pipe(lambda w: w[w["is_first"]] if "합친" in v else w)["label_conflict"].sum())
                     for v in summary.index]
summary

,윈도우,정상,공격,라벨 충돌
A 전체,28387,19730,8657,368
A 합친 뒤 (학습),9378,5751,3627,170
B 전체 (학습),8183,4681,3502,97
B 합친 뒤,7784,4495,3289,94


기법별 공격 윈도우 수 (학습에 쓰는 기준: A는 합친 뒤, B는 전체).

In [12]:
train = {"A": windows["A"][windows["A"]["is_first"]], "B": windows["B"]}
per_tech = pd.DataFrame({
    v: w[w["is_attack"]]["techniques"].str.split("|").explode().value_counts() for v, w in train.items()
}).fillna(0).astype(int).sort_values("B", ascending=False)
per_tech

,A,B
techniques,,
T1580,1870,1798
T1526,1040,935
T1496.001,464,436
T1098,97,167
T1078,57,89
T1078.004,61,62
T1530,26,51
T1686.001,45,50
T1486,21,20


같은 공격 실행이 A와 B에서 어떻게 보이는지 비교한다. 예: `ssm-send-command`는 도구가 결과를 기다리며 같은 조회를 50번 반복한다.

In [13]:
session = "stratus:aws.execution.ssm-send-command"
for v in ("A", "B"):
    w = train[v]
    w = w[(w["session"] == session) & w["is_attack"]]
    print(f"[{v}] 공격 윈도우 {len(w)}개")
    print(w["eventNames"].str.replace("DescribeInstanceInformation", "DII").head(6).to_string(index=False), "\n")

[A] 공격 윈도우 7개
                       DII > DII > DII > DII > DII
               DII > DII > DII > DII > SendCommand
               DII > DII > DII > SendCommand > DII
DII > DII > SendCommand > DII > GetCommandInvoc...
DII > SendCommand > DII > GetCommandInvocation ...
SendCommand > DII > GetCommandInvocation > GetC... 

[B] 공격 윈도우 1개
DII > SendCommand > DII > GetCommandInvocation 

